# Notebook 7 — Validação da Camada Semântica: Cobertura das Variáveis de Input

**Objetivo:** Verificar a cobertura (% de empresas com dado disponível) de cada variável contábil que será usada no cálculo de indicadores financeiros, identificando quais precisam de COALESCE.  
**Input:** `layer_02_silver.n1_dfp_cia_aberta_bp` · `n1_dfp_cia_aberta_dre` · `n1_dfp_cia_aberta_dfc`  
**Output:** Relatório de cobertura por variável (V01–V26) — sem escrita no banco

## 1. Setup — Bibliotecas, Conexão e Tabelas de Referência

In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

def create_db_engine():
    user     = quote_plus(os.getenv('DB_USER', 'postgres'))
    password = quote_plus(os.getenv('DB_PASS', 'password'))
    host     = os.getenv('DB_HOST', 'localhost')
    port     = os.getenv('DB_PORT', '5432')
    dbname   = os.getenv('DB_NAME', 'data_lake')
    return create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}')

engine = create_db_engine()

TABELA_BP  = 'layer_02_silver.n1_dfp_cia_aberta_bp'
TABELA_DRE = 'layer_02_silver.n1_dfp_cia_aberta_dre'
TABELA_DFC = 'layer_02_silver.n1_dfp_cia_aberta_dfc'

print(f'BP  -> {TABELA_BP}')
print(f'DRE -> {TABELA_DRE}')
print(f'DFC -> {TABELA_DFC}')


## 2. Função Helper de Validação — `validar_variavel()`

> Recebe um `CD_CONTA`, consulta a tabela Silver e retorna o percentual de empresas  
> com dado disponível no período mais recente. Armazena o resultado em `resultados_cobertura`.

In [ ]:
def validar_variavel(
    engine,
    var_id: str,
    conceito: str,
    tabela: str,
    contas: list,
    coalesce_zero: bool = False,
    cobertura_esperada_pct: float = 95.0
):
    """
    Para uma variável, exibe:
      1. Catálogo Semântico: CD_CONTA + DS_CONTA + ST_CONTA_FIXA + IS_LEAF
         => as linhas exatas que constituem este conceito de negócio
      2. Cobertura: qtd empresas e períodos com esta conta
      3. Faixa de valores (excluindo zeros)
    """
    contas_sql = ', '.join(f"'{c}'" for c in contas)
    print(f'\n{"="*70}')
    print(f'  {var_id} — {conceito}')
    print(f'  Conta(s): {contas}   |   Tabela: {tabela.split(".")[-1]}')
    print(f'{"="*70}')

    # ── 1. CATÁLOGO SEMÂNTICO ─────────────────────────────────────────────
    q_catalogo = f"""
    SELECT
        "CD_CONTA",
        "DS_CONTA"                          AS nome_padronizado,
        "ST_CONTA_FIXA",
        "IS_LEAF",
        COUNT(*)                            AS n_linhas,
        COUNT(DISTINCT "CNPJ_CIA")          AS n_empresas,
        COUNT(DISTINCT "DT_REFER")          AS n_periodos
    FROM {tabela}
    WHERE "CD_CONTA" IN ({contas_sql})
    GROUP BY "CD_CONTA", "DS_CONTA", "ST_CONTA_FIXA", "IS_LEAF"
    ORDER BY "CD_CONTA", n_linhas DESC;
    """

    # ── 2. COBERTURA ──────────────────────────────────────────────────────
    q_cobertura = f"""
    WITH universo AS (
        SELECT COUNT(DISTINCT "CNPJ_CIA") AS total_empresas,
               COUNT(DISTINCT ("CNPJ_CIA" || "DT_REFER"::TEXT)) AS total_empresa_periodo
        FROM {tabela}
    ),
    com_conta AS (
        SELECT COUNT(DISTINCT "CNPJ_CIA") AS empresas_com_conta,
               COUNT(DISTINCT ("CNPJ_CIA" || "DT_REFER"::TEXT)) AS ep_com_conta
        FROM {tabela}
        WHERE "CD_CONTA" IN ({contas_sql})
    )
    SELECT
        u.total_empresas,
        c.empresas_com_conta,
        ROUND((c.empresas_com_conta::NUMERIC / NULLIF(u.total_empresas,0) * 100), 1) AS pct_empresas,
        u.total_empresa_periodo,
        c.ep_com_conta,
        ROUND((c.ep_com_conta::NUMERIC / NULLIF(u.total_empresa_periodo,0) * 100), 1) AS pct_ep
    FROM universo u, com_conta c;
    """

    # ── 3. FAIXA DE VALORES ───────────────────────────────────────────────
    q_valores = f"""
    SELECT
        COUNT(*)                                         AS n_total,
        SUM(CASE WHEN "VL_CONTA_TRATADO" = 0 THEN 1 ELSE 0 END) AS n_zeros,
        MIN("VL_CONTA_TRATADO")                          AS min_valor,
        PERCENTILE_CONT(0.5) WITHIN GROUP
            (ORDER BY "VL_CONTA_TRATADO")                AS mediana,
        MAX("VL_CONTA_TRATADO")                          AS max_valor
    FROM {tabela}
    WHERE "CD_CONTA" IN ({contas_sql})
      AND "IS_LEAF" = TRUE;
    """

    with engine.connect() as conn:
        df_cat = pd.read_sql(text(q_catalogo),  conn)
        df_cob = pd.read_sql(text(q_cobertura), conn)
        df_val = pd.read_sql(text(q_valores),   conn)

    print('\n📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:')
    display(df_cat)

    pct = float(df_cob['pct_empresas'].iloc[0]) if not df_cob.empty else 0
    status_cob = '✅' if pct >= cobertura_esperada_pct else ('⚠️' if pct >= 70 else '❌')
    print(f'\n📊 COBERTURA: {status_cob}')
    display(df_cob)

    print(f'\n💰 FAIXA DE VALORES (IS_LEAF = True):')
    display(df_val)

    coalesce_msg = '  ℹ️  COALESCE(valor, 0) — ausência = empresa sem esta rubrica (tratar como zero)' if coalesce_zero else ''
    if coalesce_msg:
        print(coalesce_msg)

    return pct


# Dicionário de resultados para o resumo final
resultados_cobertura = {}
print('✅ Helper definido.')


## 3. Variáveis do Balanço Patrimonial (BP)

> V01 a V16: contas do Ativo, Passivo e Patrimônio Líquido usadas nos indicadores de liquidez, estrutura e Fleuriet.

### V01 — Ativo Total (AT)
> `CD_CONTA = '1'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V01', 'Ativo Total (AT)', TABELA_BP, ['1'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V01'] = pct


### V02 — Ativo Circulante (AC)
> `CD_CONTA = '1.01'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V02', 'Ativo Circulante (AC)', TABELA_BP, ['1.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V02'] = pct


### V03 — Ativo Não Circulante (ANC)
> `CD_CONTA = '1.02'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V03', 'Ativo Não Circulante (ANC)', TABELA_BP, ['1.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V03'] = pct


### V04 — Realizável a Longo Prazo (RLP / ELP)
> `CD_CONTA = '1.02.01'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V04', 'Realizável a Longo Prazo (RLP / ELP)', TABELA_BP, ['1.02.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V04'] = pct


### V05 — Ativo Imobilizado
> `CD_CONTA = '1.02.03'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V05', 'Ativo Imobilizado', TABELA_BP, ['1.02.03'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V05'] = pct


### V06 — Estoques
> `CD_CONTA = '1.01.04'` | `COALESCE(valor, 0)`

In [ ]:
pct = validar_variavel(engine, 'V06', 'Estoques', TABELA_BP, ['1.01.04'], coalesce_zero=True, cobertura_esperada_pct=85.0)
resultados_cobertura['V06'] = pct


### V07 — Contas a Receber CP
> `CD_CONTA = '1.01.03'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V07', 'Contas a Receber CP', TABELA_BP, ['1.01.03'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V07'] = pct


### V09 — Fornecedores
> `CD_CONTA = '2.01.02'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V09', 'Fornecedores', TABELA_BP, ['2.01.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V09'] = pct


### V10 — Passivo Circulante (PC)
> `CD_CONTA = '2.01'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V10', 'Passivo Circulante (PC)', TABELA_BP, ['2.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V10'] = pct


### V11 — Passivo Não Circulante / ELP
> `CD_CONTA = '2.02'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V11', 'Passivo Não Circulante / ELP', TABELA_BP, ['2.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V11'] = pct


### V12 — Patrimônio Líquido (PL)
> `CD_CONTA = '2.03'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V12', 'Patrimônio Líquido (PL)', TABELA_BP, ['2.03'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V12'] = pct


### V13 — Empréstimos e Financ. CP
> `CD_CONTA = '2.01.04'` | `COALESCE(valor, 0)`

In [ ]:
pct = validar_variavel(engine, 'V13', 'Empréstimos e Financ. CP', TABELA_BP, ['2.01.04'], coalesce_zero=True, cobertura_esperada_pct=95.0)
resultados_cobertura['V13'] = pct


### V14 — Empréstimos e Financ. LP
> `CD_CONTA = '2.02.01'` | `COALESCE(valor, 0)`

In [ ]:
pct = validar_variavel(engine, 'V14', 'Empréstimos e Financ. LP', TABELA_BP, ['2.02.01'], coalesce_zero=True, cobertura_esperada_pct=95.0)
resultados_cobertura['V14'] = pct


### V22 — Disponibilidades / Caixa BP
> `CD_CONTA = '1.01.01'` | `sem COALESCE`

In [ ]:
pct = validar_variavel(engine, 'V22', 'Disponibilidades / Caixa BP', TABELA_BP, ['1.01.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V22'] = pct


### V15 — Ativo Circulante Financeiro (ACF) — Fleuriet
> `CD_CONTA IN ('1.01.01', '1.01.02')` | COALESCE para `1.01.02`

In [ ]:
pct = validar_variavel(engine, 'V15', 'ACF Fleuriet (Caixa + Aplic.Fin.)', TABELA_BP,
                       ['1.01.01', '1.01.02'], coalesce_zero=True, cobertura_esperada_pct=85.0)
resultados_cobertura['V15'] = pct


### V16 — Passivo Circulante Financeiro (PCF) — Fleuriet
> Idêntico a V13: `CD_CONTA = '2.01.04'`

In [ ]:
# V16 é o mesmo CD_CONTA de V13 — apenas documenta o papel semântico diferente (PCF no Fleuriet)
pct = validar_variavel(engine, 'V16', 'PCF Fleuriet (Empréstimos CP)', TABELA_BP,
                       ['2.01.04'], coalesce_zero=True, cobertura_esperada_pct=95.0)
resultados_cobertura['V16'] = pct


## 4. Variáveis da DRE

> V17 a V26: contas de resultado usadas nos indicadores de lucratividade, margens e LPA.

### V17 — Receita Líquida de Vendas
> `CD_CONTA = '3.01'`

In [ ]:
pct = validar_variavel(engine, 'V17', 'Receita Líquida de Vendas', TABELA_DRE, ['3.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V17'] = pct


### V18 — Custo dos Bens/Serviços (CPV)
> `CD_CONTA = '3.02'`

In [ ]:
pct = validar_variavel(engine, 'V18', 'Custo dos Bens/Serviços (CPV)', TABELA_DRE, ['3.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V18'] = pct


### V19 — Lucro Bruto (Resultado Bruto)
> `CD_CONTA = '3.03'`

In [ ]:
pct = validar_variavel(engine, 'V19', 'Lucro Bruto (Resultado Bruto)', TABELA_DRE, ['3.03'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V19'] = pct


### V20 — Lucro Operacional (EBIT)
> `CD_CONTA = '3.05'`

In [ ]:
pct = validar_variavel(engine, 'V20', 'Lucro Operacional (EBIT)', TABELA_DRE, ['3.05'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V20'] = pct


### V21 — Lucro Líquido do Exercício
> `CD_CONTA = '3.11'`

In [ ]:
pct = validar_variavel(engine, 'V21', 'Lucro Líquido do Exercício', TABELA_DRE, ['3.11'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V21'] = pct


### V25 — LPA Básico ON
> `CD_CONTA = '3.99.01.01'`

In [ ]:
pct = validar_variavel(engine, 'V25', 'LPA Básico ON', TABELA_DRE, ['3.99.01.01'], coalesce_zero=False, cobertura_esperada_pct=80.0)
resultados_cobertura['V25'] = pct


### V26 — LPA Diluído ON
> `CD_CONTA = '3.99.02.01'`

In [ ]:
pct = validar_variavel(engine, 'V26', 'LPA Diluído ON', TABELA_DRE, ['3.99.02.01'], coalesce_zero=False, cobertura_esperada_pct=80.0)
resultados_cobertura['V26'] = pct


### V26b — Cobertura COALESCE(V26, V25): estratégia real do indicador LPA Diluído
> V26 isolado = 67.6% por empresa-período. Com fallback V25: **86.7%**. Os 13.3% restantes não têm LPA de nenhum tipo (estrutural).

In [ ]:
q_coalesce_lpa = """
WITH v25 AS (
    SELECT "CNPJ_CIA", "DT_REFER", "VL_CONTA_TRATADO" AS lpa_basico
    FROM layer_02_silver.n1_dfp_cia_aberta_dre
    WHERE "CD_CONTA" = '3.99.01.01'
      AND "FLAG_RECONSTRUCAO" = False
      AND ABS("VL_CONTA_TRATADO") <= 10000
),
v26 AS (
    SELECT "CNPJ_CIA", "DT_REFER", "VL_CONTA_TRATADO" AS lpa_diluido
    FROM layer_02_silver.n1_dfp_cia_aberta_dre
    WHERE "CD_CONTA" = '3.99.02.01'
      AND "FLAG_RECONSTRUCAO" = False
      AND ABS("VL_CONTA_TRATADO") <= 10000
),
universo AS (
    SELECT DISTINCT "CNPJ_CIA", "DT_REFER"
    FROM layer_02_silver.n1_dfp_cia_aberta_dre
    WHERE "CD_CONTA" = '3.01'
),
combinado AS (
    SELECT
        u."CNPJ_CIA",
        u."DT_REFER",
        CASE
            WHEN v26.lpa_diluido IS NOT NULL THEN 'diluido_direto'
            WHEN v25.lpa_basico  IS NOT NULL THEN 'fallback_basico'
            ELSE 'sem_lpa'
        END AS origem
    FROM universo u
    LEFT JOIN v25 ON v25."CNPJ_CIA" = u."CNPJ_CIA" AND v25."DT_REFER" = u."DT_REFER"
    LEFT JOIN v26 ON v26."CNPJ_CIA" = u."CNPJ_CIA" AND v26."DT_REFER" = u."DT_REFER"
)
SELECT
    origem,
    COUNT(*) AS qtd_empresa_periodo,
    ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER())::NUMERIC, 1) AS pct
FROM combinado
GROUP BY origem
ORDER BY qtd_empresa_periodo DESC;
"""

with engine.connect() as conn:
    df_coalesce = pd.read_sql(text(q_coalesce_lpa), conn)

print('📊 COBERTURA REAL — COALESCE(V26, V25) por empresa-período:')
display(df_coalesce)

cobertura_coalesce_ep = float(df_coalesce.loc[df_coalesce['origem'] != 'sem_lpa', 'pct'].sum())
print(f'\n✅ Cobertura efetiva COALESCE(V26, V25): {cobertura_coalesce_ep:.1f}% dos empresa-período')
print(f'   ├── diluído direto  : {df_coalesce.loc[df_coalesce["origem"]=="diluido_direto","pct"].values[0]}%')
print(f'   ├── fallback básico : {df_coalesce.loc[df_coalesce["origem"]=="fallback_basico","pct"].values[0]}%')
print(f'   └── sem LPA (estrutural): {df_coalesce.loc[df_coalesce["origem"]=="sem_lpa","pct"].values[0]}% — irreducível')

resultados_cobertura['V26_COALESCE_EP'] = cobertura_coalesce_ep


## 5. Variáveis da DFC

> V23: saldo final de caixa (`6.05.02`) — conceito CPC 03, inclui equivalentes até 90 dias.

### V23 — Saldo Final de Caixa (para Dívida Líquida / EV)
> `CD_CONTA = '6.05.02'` | DFC (CPC 03 — inclui CDBs/LFTs ≤90d)

In [ ]:
pct = validar_variavel(engine, 'V23', 'Saldo Final Caixa DFC (6.05.02)', TABELA_DFC,
                       ['6.05.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V23'] = pct


## 6. Resumo Executivo — Cobertura por Variável

> Consolida todos os resultados de `resultados_cobertura` em um DataFrame ordenado.  
> Variáveis com cobertura < 80% devem usar `COALESCE(valor, 0)` no cálculo dos indicadores.

In [ ]:
df_resumo = pd.DataFrame(
    [(k, v) for k, v in sorted(resultados_cobertura.items())],
    columns=['Variavel', 'Cobertura_Pct']
)
df_resumo['Status'] = df_resumo['Cobertura_Pct'].apply(
    lambda x: '✅ OK' if x >= 95 else ('⚠️ Parcial' if x >= 70 else '❌ Baixa')
)
df_resumo['Observacao'] = df_resumo['Variavel'].map({
    'V06': 'Serviços puros sem estoque — COALESCE(,0)',
    'V13': 'Empresa sem dívida CP — COALESCE(,0)',
    'V14': 'Empresa sem dívida LP — COALESCE(,0)',
    'V25': 'LPA Básico ON — 86.5% por empresa-período',
    'V26': 'LPA Diluído standalone — usar COALESCE(V26,V25)',
    'V26_COALESCE_EP': '✅ Cobertura REAL usada no dash (por empresa-período)',
})
df_resumo = df_resumo.sort_values('Cobertura_Pct', ascending=False)

print('\n📋 RESUMO — Cobertura das Variáveis de Input')
display(df_resumo.style.bar(subset=['Cobertura_Pct'], color='#5fba7d'))

ok   = (df_resumo['Cobertura_Pct'] >= 95).sum()
parc = ((df_resumo['Cobertura_Pct'] >= 70) & (df_resumo['Cobertura_Pct'] < 95)).sum()
baixa= (df_resumo['Cobertura_Pct'] < 70).sum()
print(f'\n✅ OK (≥95%): {ok}  |  ⚠️ Parcial (70-95%): {parc}  |  ❌ Baixa (<70%): {baixa}')
print(f'\n⚠️  V26 standalone = 82.2% (por empresa) / 67.6% (por empresa-período)')
print(f'✅  COALESCE(V26,V25) = 86.7% por empresa-período — este é o número que vai para o dash')
print(f'📌  13.3% sem LPA são estruturais (holdings/empresas sem ação ordinária) — irreducível')
